# Hybrid Recommender — Content-Based + Collaborative Filtering
**Dataset:** Amazon Electronics (SQLite)

Pipeline:
1. Carregar dados
2. Gerar ratings sintéticos (se não existir tabela de ratings)
3. Content-Based Filtering (CBF)
4. Collaborative Filtering (CF) — item-item
5. Hybrid Recommender (weighted)

## 1. Importações e ligação à base de dados

In [12]:
import sqlite3, pandas as pd
conn = sqlite3.connect("../fastapi_recommender/amazon_electronics.db")
print(pd.read_sql("SELECT * FROM ratings LIMIT 3", conn))
conn.close()

  review_id  product_id                                            user_id  \
0         1  B07JW9H4J1  AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...   
1         2  B098NS6PVG  AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...   
2         3  B096MSW6CT  AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...   

   rating                                       review_title  \
0     4.2  Satisfied,Charging is really fast,Value for mo...   
1     4.0  A Good Braided Cable for Your Type C Device,Go...   
2     3.9  Good speed for earlier versions,Good Product,W...   

                                      review_content Used_Device Day_of_Week  
0  Looks durable Charging is fine tooNo complains...     Desktop    Saturday  
1  I ordered this cable to connect my phone to An...      Mobile     Tuesday  
2  Not quite durable and sturdy,https://m.media-a...      Tablet     Tuesday  


In [13]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity

DB_PATH = "../fastapi_recommender/amazon_electronics.db"

conn = sqlite3.connect(DB_PATH)

# Ver todas as tabelas disponíveis
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)
print("Tabelas na base de dados:", tables["name"].tolist())

df = pd.read_sql_query("SELECT * FROM products", conn)
conn.close()

print(f"\nProdutos carregados: {len(df)}")
df.head()

Tabelas na base de dados: ['users', 'products', 'ratings']

Produtos carregados: 1351


,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating_count,about_product,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,399.0,1.099,0.64,24269,High Compatibility : Compatible With iPhone 12...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,199.0,349.000,0.43,43994,"Compatible with all Type C enabled devices, be...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,199.0,1.899,0.90,7928,【 Fast Charger& Data Sync】-With built-in safet...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,699.000,0.53,94363,The boAt Deuce USB 300 2 in 1 cable is compati...,https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,154.0,399.000,0.61,16905,[CHARGE & SYNC FUNCTION]- This cable comes wit...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


## 2. Gerar ratings sintéticos

O dataset de produtos não tem ratings individuais por utilizador.
Geramos ratings sintéticos realistas usando o `rating_count` como proxy de popularidade:
- Produtos mais populares têm maior probabilidade de serem avaliados
- Os ratings são amostrados com distribuição que favorece valores altos (como em dados reais)

In [14]:
def generate_synthetic_ratings(df, n_users=200, seed=42):
    """
    Gera ratings sintéticos com base na popularidade dos produtos (rating_count).
    Produtos com mais avaliações têm maior probabilidade de serem selecionados.
    """
    np.random.seed(seed)

    product_ids = df["product_id"].values

    # Normalizar rating_count para usar como probabilidade de seleção
    counts = df["rating_count"].fillna(1).values.astype(float)
    probs = counts / counts.sum()

    rows = []
    for user_id in range(1, n_users + 1):
        # Cada utilizador avalia entre 3 e 15 produtos
        n_ratings = np.random.randint(3, 16)

        # Selecionar produtos com probabilidade proporcional à popularidade
        rated_products = np.random.choice(
            product_ids,
            size=min(n_ratings, len(product_ids)),
            replace=False,
            p=probs
        )

        for product_id in rated_products:
            # Rating entre 1 e 5, com distribuição realista (maioria positiva)
            rating = np.random.choice(
                [1, 2, 3, 4, 5],
                p=[0.05, 0.10, 0.20, 0.35, 0.30]
            )
            rows.append({"user_id": user_id, "product_id": product_id, "rating": rating})

    return pd.DataFrame(rows)


# Verificar se já existe uma tabela de ratings na DB
conn = sqlite3.connect(DB_PATH)
existing_tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)["name"].tolist()

if "ratings" in existing_tables:
    print("Tabela 'ratings' encontrada na DB — a usar dados reais.")
    ratings = pd.read_sql_query("SELECT * FROM ratings", conn)
else:
    print("Tabela 'ratings' não encontrada — a gerar ratings sintéticos.")
    ratings = generate_synthetic_ratings(df, n_users=200)

conn.close()

print(f"\nTotal de ratings: {len(ratings)}")
print(f"Utilizadores únicos: {ratings['user_id'].nunique()}")
print(f"Produtos avaliados: {ratings['product_id'].nunique()}")
ratings.head()

Tabela 'ratings' encontrada na DB — a usar dados reais.

Total de ratings: 5963
Utilizadores únicos: 1194
Produtos avaliados: 1351


,review_id,product_id,user_id,rating,review_title,review_content,Used_Device,Day_of_Week
0,1,B07JW9H4J1,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...",4.2,"Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,Desktop,Saturday
1,2,B098NS6PVG,"AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...",4.0,"A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,Mobile,Tuesday
2,3,B096MSW6CT,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...",3.9,"Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",Tablet,Tuesday
3,4,B08HDJ86NZ,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...",4.2,"Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",Desktop,Friday
4,5,B08CF3B7N1,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...",4.2,"As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",Mobile,Thursday


## 3. Content-Based Filtering (CBF)
Usa categoria e preço para encontrar produtos similares.

In [15]:
# Preparar features de conteúdo
df_cbf = df.copy()
df_cbf["category"] = df_cbf["category"].str.split(r"[|&]")
df_cbf["category"] = df_cbf["category"].apply(lambda x: list(set(x)))

mlb = MultiLabelBinarizer()
category_matrix = mlb.fit_transform(df_cbf["category"])

scaler = MinMaxScaler()
price_scaled = scaler.fit_transform(df_cbf[["discounted_price"]])

feature_matrix = np.hstack((category_matrix, price_scaled))
content_similarity = cosine_similarity(feature_matrix)

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Content similarity matrix: {content_similarity.shape}")

Feature matrix shape: (1351, 362)
Content similarity matrix: (1351, 1351)


In [16]:
# Mapa product_id -> índice (partilhado por CBF e CF)
product_index_map = pd.Series(df.index, index=df["product_id"]).to_dict()


def get_cbf_scores(product_id, k=10):
    """
    Retorna dict {product_id: content_similarity_score} para os k produtos mais similares.
    """
    if product_id not in product_index_map:
        return {}

    idx = product_index_map[product_id]
    scores = list(enumerate(content_similarity[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)[1:k + 1]

    return {df.iloc[i]["product_id"]: float(score) for i, score in scores}


# Teste
example_pid = df["product_id"].iloc[0]
print(f"CBF para produto '{example_pid}':")
get_cbf_scores(example_pid, k=5)

CBF para produto 'B07JW9H4J1':


{'B07JW1Y6XV': 1.0,
 'B09RWZRCP1': 1.0,
 'B07LGT55SJ': 1.0,
 'B09QGZFBPM': 1.0,
 'B09RX1FK54': 1.0}

## 4. Collaborative Filtering (CF) — Item-Item

Cria a user-item matrix e calcula similaridade entre produtos com base nos padrões de avaliação.

In [17]:
# 4.1 Construir a user-item matrix
# Apenas produtos que existem em df (garantir consistência)
valid_products = set(df["product_id"])
ratings_clean = ratings[ratings["product_id"].isin(valid_products)].copy()

user_item_matrix = ratings_clean.pivot_table(
    index="user_id",
    columns="product_id",
    values="rating"
)

print(f"User-item matrix: {user_item_matrix.shape} (utilizadores x produtos)")
print(f"Sparsidade: {user_item_matrix.isna().sum().sum() / user_item_matrix.size:.1%} valores em falta")
user_item_matrix.head()

User-item matrix: (1194, 1350) (utilizadores x produtos)
Sparsidade: 99.6% valores em falta


product_id,B002PD61Y4,B002SZEOLG,B003B00484,B003L62T7W,B004IO5BMQ,B005FYNT3G,B005LJQMCK,B005LJQMZC,B006LW0WDQ,B0073QGKAS,...,B0BP18W8TM,B0BP7XLX48,B0BP89YBC1,B0BPBG712X,B0BPBXNQQT,B0BPCJM7TB,B0BPJBTB3F,B0BQ3K23Y1,B0BQRJ3C47,B0BR4F878Q
user_id,,,,,,,,,,,,,,,,,,,,,
"AE22Y3KIS7SE6LI3HE2VS6WWPU4Q,AHWEYO2IJ5I5GDWZAHJK6NGYHFMA,AGYURQ3476BNT4D2O46THXEUY3SA,AFPMBSBIEX45OQ6UCQWPDG55GWLQ,AGWJU3WUQBDQYPSYAJSR3AKBLCOA,AEOVUNFCIFV223O536GVW5JHZKOA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"AE23RS3W7GZO7LHYKJU6KSKVM4MQ,AEQUNEY6GQOTEGUMS6KRUEYNXJSQ,AGYPIE5BICV44WEEEPJVEFQOCJSQ,AFR7CEQKWZE53IHHOWBIPAMYKL4Q,AGBV7FBP4SEITF6UKRFKTV7O32IA,AHQVOY54QKPIQZIJ57JKCGQPVV3Q,AEMCVRRD3XQRGFHC2VFCXHJEMESQ,AFBWXU7DUWCIK5MRDCLBXWTWN7ZQ",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
AE242TR3GQ6TYC6W4SJ5UYYKBTYQ,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"AE27UOZENYSWCQVQRRUQIV2ZM7VA,AGMYSLV6NNOAYES25JDTJPCZY47A,AFHS33MWRQGSS64EETZJGCBWXXXA,AHYXZVXUY3QTBP7IBFIUBSZVH2XQ,AH2SHWYEWDAK6A5Y2ZBEMZ2KIG3A,AEYMOGP2CYRKYZ7TIDNLGR5QPZ4Q,AGPGDCCXPI3EACMNJKBCNT57DVFA,AFPBMRYRSMD3PP3CBKLFF7EKOCXA",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
"AE2JTMRKTUOIVIZWS2WDGTMNTU4Q,AF4QXCB32VC2DVE7O3DGFNQVFFNQ,AGAFYHMPFGVPR3MOS4QAZLAWPW3A,AGNNWLEF6V57TKIFJM7SWHNFAIQQ,AFVIPOPKMOCVCX3CMXUJHMWDIMGA,AH6MFUU725GG4KA3XTALSTU2ILHA,AGQYTSKE2UBYARZYRBADQMX6BJPQ,AG7F66F724JZ2HIJQY7NOU5M5D2Q",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [18]:
# 4.2 Preencher NaN com 0 e calcular similaridade item-item
user_item_filled = user_item_matrix.fillna(0)

# .T porque queremos similaridade entre produtos (colunas), não utilizadores (linhas)
cf_similarity_matrix = cosine_similarity(user_item_filled.T)

cf_similarity_df = pd.DataFrame(
    cf_similarity_matrix,
    index=user_item_filled.columns,
    columns=user_item_filled.columns
)

print(f"CF similarity matrix: {cf_similarity_df.shape}")
cf_similarity_df.head()

CF similarity matrix: (1350, 1350)


product_id,B002PD61Y4,B002SZEOLG,B003B00484,B003L62T7W,B004IO5BMQ,B005FYNT3G,B005LJQMCK,B005LJQMZC,B006LW0WDQ,B0073QGKAS,...,B0BP18W8TM,B0BP7XLX48,B0BP89YBC1,B0BPBG712X,B0BPBXNQQT,B0BPCJM7TB,B0BPJBTB3F,B0BQ3K23Y1,B0BQRJ3C47,B0BR4F878Q
product_id,,,,,,,,,,,,,,,,,,,,,
B002PD61Y4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B002SZEOLG,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B003B00484,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B003L62T7W,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
B004IO5BMQ,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [19]:
def get_cf_scores(rated_products, k=10):
    """
    Dado um dict {product_id: rating}, retorna scores CF para produtos candidatos.
    rated_products: dict {product_id: rating}
    """
    scores = {}

    for product_id, rating in rated_products.items():
        if product_id not in cf_similarity_df.columns:
            continue

        similar = cf_similarity_df[product_id].drop(index=product_id, errors="ignore")
        top_similar = similar.nlargest(k)

        for similar_pid, sim_score in top_similar.items():
            if similar_pid in rated_products:
                continue
            scores[similar_pid] = scores.get(similar_pid, 0) + float(sim_score) * rating

    return scores


# Teste
test_ratings = {df["product_id"].iloc[0]: 4.5, df["product_id"].iloc[5]: 3.0}
cf_result = get_cf_scores(test_ratings, k=10)
print(f"CF produziu {len(cf_result)} candidatos")
sorted(cf_result.items(), key=lambda x: x[1], reverse=True)[:5]

CF produziu 16 candidatos


[('B07JGDB5M1', 4.5),
 ('B07JH1C41D', 4.5),
 ('B07JH1CBGW', 4.5),
 ('B07JW1Y6XV', 4.5),
 ('B07LGT55SJ', 4.5)]

## 5. Hybrid Recommender (Weighted)

Combina CBF e CF com pesos ajustáveis:
```
final_score = weight_cf * cf_score + weight_cbf * cbf_score
```

In [20]:
def normalize_scores(scores_dict):
    """Normaliza scores para [0, 1]."""
    if not scores_dict:
        return {}
    max_val = max(scores_dict.values())
    if max_val == 0:
        return scores_dict
    return {k: v / max_val for k, v in scores_dict.items()}


def hybrid_recommender(rated_products, n=5, weight_cf=0.6, weight_cbf=0.4):
    """
    Recomendador híbrido: combina Collaborative Filtering e Content-Based Filtering.

    Parâmetros:
        rated_products : dict {product_id: rating} — produtos já avaliados pelo utilizador
        n              : número de recomendações a devolver
        weight_cf      : peso do CF no score final (default 0.6)
        weight_cbf     : peso do CBF no score final (default 0.4)
    """
    k = n * 3  # candidatos intermédios

    # --- CF scores ---
    raw_cf = get_cf_scores(rated_products, k=k)
    norm_cf = normalize_scores(raw_cf)

    # --- CBF scores (agregar sobre todos os produtos avaliados) ---
    raw_cbf = {}
    for product_id, rating in rated_products.items():
        for pid, score in get_cbf_scores(product_id, k=k).items():
            if pid in rated_products:
                continue
            raw_cbf[pid] = raw_cbf.get(pid, 0) + score * rating
    norm_cbf = normalize_scores(raw_cbf)

    # --- Combinar todos os candidatos ---
    all_candidates = set(norm_cf.keys()) | set(norm_cbf.keys())

    final_scores = {}
    for pid in all_candidates:
        cf_score  = norm_cf.get(pid, 0)
        cbf_score = norm_cbf.get(pid, 0)
        final_scores[pid] = weight_cf * cf_score + weight_cbf * cbf_score

    # --- Top N ---
    top_n_ids = sorted(final_scores, key=lambda x: final_scores[x], reverse=True)[:n]

    result = df[df["product_id"].isin(top_n_ids)][[
        "product_id", "product_name", "category", "discounted_price"
    ]].copy()

    result["hybrid_score"]  = result["product_id"].map(final_scores)
    result["cf_score"]      = result["product_id"].map(lambda x: norm_cf.get(x, 0))
    result["cbf_score"]     = result["product_id"].map(lambda x: norm_cbf.get(x, 0))

    return result.sort_values("hybrid_score", ascending=False).reset_index(drop=True)

## 6. Testar o Hybrid Recommender

In [21]:
# Produtos avaliados pelo utilizador (product_id: rating)
rated_products = {
    "B0789LZTCJ": 4.2,
    "B094JNXNPV": 3.5
}

recommendations = hybrid_recommender(
    rated_products=rated_products,
    n=5,
    weight_cf=0.6,
    weight_cbf=0.4
)

recommendations

,product_id,product_name,category,discounted_price,hybrid_score,cf_score,cbf_score
0,B08HDH26JX,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,299.0,1.0,1.0,1.0
1,B07CRL2GY6,boAt Rugged V3 Braided Micro USB Cable (Pearl ...,Computers&Accessories|Accessories&Peripherals|...,299.0,1.0,1.0,1.0
2,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,329.0,0.6,1.0,0.0
3,B077Z65HSD,boAt A400 USB Type-C to USB-A 2.0 Male Data Ca...,Computers&Accessories|Accessories&Peripherals|...,299.0,0.4,0.0,1.0
4,B00NH13Q8W,AmazonBasics USB 2.0 Extension Cable for Perso...,Computers&Accessories|Accessories&Peripherals|...,299.0,0.4,0.0,1.0


In [22]:
# Comparar: só CBF vs só CF vs Hybrid
print("=" * 60)
print("COMPARAÇÃO: CBF puro vs CF puro vs Hybrid")
print("=" * 60)

print("\n📦 Só Content-Based (weight_cf=0, weight_cbf=1):")
print(hybrid_recommender(rated_products, n=5, weight_cf=0.0, weight_cbf=1.0)[["product_name", "cbf_score"]])

print("\n👥 Só Collaborative Filtering (weight_cf=1, weight_cbf=0):")
print(hybrid_recommender(rated_products, n=5, weight_cf=1.0, weight_cbf=0.0)[["product_name", "cf_score"]])

print("\n⚡ Hybrid (weight_cf=0.6, weight_cbf=0.4):")
print(hybrid_recommender(rated_products, n=5, weight_cf=0.6, weight_cbf=0.4)[["product_name", "hybrid_score"]])

COMPARAÇÃO: CBF puro vs CF puro vs Hybrid

📦 Só Content-Based (weight_cf=0, weight_cbf=1):
                                        product_name  cbf_score
0  boAt A400 USB Type-C to USB-A 2.0 Male Data Ca...        1.0
1  Amazon Basics USB 3.0 Cable - A Male to Micro ...        1.0
2  AmazonBasics USB 2.0 Extension Cable for Perso...        1.0
3            Boat A 350 Type C Cable 1.5m(Jet Black)        1.0
4  Storite USB 2.0 A to Mini 5 pin B Cable for Ex...        1.0

👥 Só Collaborative Filtering (weight_cf=1, weight_cbf=0):
                                        product_name  cf_score
0  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...  1.000000
1  boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...  1.000000
2  boAt Rugged V3 Braided Micro USB Cable (Pearl ...  1.000000
3  FLiX (Beetel USB to Micro USB PVC Data Sync & ...  0.147295
4  Robustrion [Anti-Scratch] & [Smudge Proof] [S ...  0.057054

⚡ Hybrid (weight_cf=0.6, weight_cbf=0.4):
                                        produ